In [1]:
"""Baselines + plots for the full CIFAR-100 (100 classes),

Baselines: Decision Tree & Logistic Regression (both on PCA features of the raw pixels).
"""
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

warnings.filterwarnings('ignore', category=UserWarning)

# Predicted class ids for the 10,000 test images, in test-set order (the test_ds is not shuffled):
extra_preds = {}
extra_acc = {}

VAL_SPLIT, SEED, N_PCA = 0.1, 42, 256

CLASS_NAMES = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed',
    'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy',
    'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle',
    'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud',
    'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl',
    'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower',
    'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree',
    'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange',
    'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree',
    'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit',
    'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea',
    'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail',
    'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper',
    'table', 'tank', 'telephone', 'television', 'tiger', 'tractor',
    'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale',
    'willow_tree', 'wolf', 'woman', 'worm',
]

SUPERCLASS_NAMES = [
    'aquatic_mammals', 'fish', 'flowers', 'food_containers', 'fruit_and_vegetables', 'household_electrical_devices',
    'household_furniture', 'insects', 'large_carnivores', 'large_man-made_outdoor_things', 'large_natural_outdoor_scenes', 'large_omnivores_and_herbivores',
    'medium_mammals', 'non-insect_invertebrates', 'people', 'reptiles', 'small_mammals', 'trees',
    'vehicles_1', 'vehicles_2',
]

# ---- data (same loader + 90/10 split ) -----------------------------------
def load_cifar100():
    from tensorflow import keras
    (xtr, ytr), (xte, yte) = keras.datasets.cifar100.load_data(label_mode='fine')
    (_, ctr), (_, cte) = keras.datasets.cifar100.load_data(label_mode='coarse')
    return xtr, ytr.ravel(), ctr.ravel(), xte, yte.ravel(), cte.ravel()

x_full, y_full, c_full, x_test, y_test, c_test = load_cifar100()
fine_to_coarse = np.zeros(100, dtype=int)
fine_to_coarse[y_full] = c_full                           # each fine class belongs to exactly one superclass

idx = np.random.default_rng(SEED).permutation(len(x_full))
n_val = int(len(idx) * VAL_SPLIT)
val_idx, train_idx = idx[:n_val], idx[n_val:]
flat = lambda x: x.reshape(len(x), -1).astype('float32') / 255.0
X_tr, y_tr = flat(x_full[train_idx]), y_full[train_idx]
X_va, y_va = flat(x_full[val_idx]),   y_full[val_idx]
X_te       = flat(x_test)
print(f"train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}")

# PCA (fit on train only) shrinks 3072 raw pixels to 256 features: much faster, about the same accuracy
t0 = time.time()
scaler = StandardScaler().fit(X_tr)
pca = PCA(N_PCA, whiten=True, random_state=SEED).fit(scaler.transform(X_tr))
P_tr, P_va, P_te = (pca.transform(scaler.transform(X)) for X in (X_tr, X_va, X_te))
print(f"PCA {N_PCA} comps, explains {100 * pca.explained_variance_ratio_.sum():.0f}% of variance ({time.time() - t0:.0f}s)")

# ---- baselines ----------------------------------------------------------------------------------
# Hyperparameters were picked once on the validation set and frozen:
#   Logistic Regression C in {0.1, 0.01, 0.001} -> 0.01      Decision Tree max_depth in {8, 12, 16} -> 12
models = {
    'Decision Tree':         DecisionTreeClassifier(max_depth=12, random_state=SEED),
}
results, preds = {}, {}
for name, m in models.items():
    t0 = time.time()
    m.fit(P_tr, y_tr)
    results[name] = {s: 100 * accuracy_score(y, m.predict(P))
                     for s, P, y in [('train', P_tr, y_tr), ('val', P_va, y_va), ('test', P_te, y_test)]}
    preds[name] = m.predict(P_te)
    r = results[name]
    print(f"{name:<24} train {r['train']:5.1f}%  val {r['val']:5.1f}%  test {r['test']:5.1f}%  ({time.time() - t0:.0f}s)")

all_preds = {**preds, **extra_preds}
test_acc = {n: results[n]['test'] for n in results}
test_acc.update({n: 100 * accuracy_score(y_test, p) for n, p in extra_preds.items()})
test_acc.update(extra_acc)
chance = 100 / 100

# Summary: fine-class accuracy and 'right superclass' accuracy
print(f"\n{'model':<24}{'test acc':>10}{'superclass acc':>16}")
for n, p in all_preds.items():
    print(f"{n:<24}{100 * accuracy_score(y_test, p):>9.1f}%{100 * np.mean(fine_to_coarse[p] == c_test):>15.1f}%")

# ---- plot 1: test accuracy, every model ---------------------------------------------------------
names = list(test_acc)
vals = [test_acc[n] for n in names]
colors = ['#9e9e9e' if n.startswith('Dummy') else '#6baed6' if n in results else '#d62728' for n in names]
fig, ax = plt.subplots(figsize=(9, 2.8 + 0.4 * len(names)))
bars = ax.barh(names[::-1], vals[::-1], color=colors[::-1])
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.axvline(chance, color='gray', linestyle=':', label=f'Chance ({chance:.0f}%)')
ax.set_xlim(0, max(vals) * 1.15); ax.set_xlabel('Test accuracy (%)')
ax.set_title('CIFAR-100 test accuracy: baselines', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout(); plt.savefig('cifar100_baseline_test_accuracy.png', dpi=150, bbox_inches='tight'); plt.show()

# ---- plot 2: train / val / test per baseline (overfitting gap) ----------------------------------
fig, ax = plt.subplots(figsize=(9, 4.5))
x, w = np.arange(len(results)), 0.26
for i, (split, c) in enumerate([('train', '#1f77b4'), ('val', '#ff7f0e'), ('test', '#2ca02c')]):
    b = ax.bar(x + (i - 1) * w, [results[n][split] for n in results], w, label=split, color=c)
    ax.bar_label(b, fmt='%.1f', padding=2, fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(list(results))
ax.axhline(chance, color='gray', linestyle=':', label='Chance (1%)')
ax.set_ylabel('Accuracy (%)'); ax.set_title('Baseline train / val / test accuracy', fontweight='bold')
ax.legend(); ax.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout(); plt.savefig('cifar100_baseline_train_val_test.png', dpi=150, bbox_inches='tight'); plt.show()

# ---- plot 3: 100x100 confusion matrix (rows = true, cols = predicted, row-normalized) -----------
best_base = max((n for n in results if not n.startswith('Dummy')), key=lambda n: results[n]['test'])
panels = [(f'{best_base} (best baseline)', preds[best_base])]
if extra_preds:
    top = max(extra_preds, key=lambda n: test_acc[n]); panels.append((top, extra_preds[top]))
fig, axes = plt.subplots(1, len(panels), figsize=(13 * len(panels), 12), squeeze=False)
for ax, (title, p) in zip(axes[0], panels):
    cm = confusion_matrix(y_test, p, labels=range(100), normalize='true')
    sns.heatmap(cm, cmap='viridis', vmin=0, vmax=1, ax=ax, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar_kws={'label': 'Fraction of true class', 'shrink': 0.7})
    ax.set_title(f"{title}: {100 * accuracy_score(y_test, p):.1f}%", fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='both', labelsize=4)
plt.tight_layout(); plt.savefig('cifar100_baseline_confusion.png', dpi=150, bbox_inches='tight'); plt.show()

# ---- plot 4: accuracy per superclass (20 groups of 5 related classes) ---------------------------
def superclass_acc(p):
    return np.array([100 * np.mean(p[c_test == c] == y_test[c_test == c]) for c in range(20)])
series = {n: superclass_acc(all_preds[n]) for n in [best_base, *extra_preds]}
order = np.argsort(series[best_base])
fig, ax = plt.subplots(figsize=(10, 7))
h = 0.8 / len(series)
for i, (n, a) in enumerate(series.items()):
    ax.barh(np.arange(20) + (i - (len(series) - 1) / 2) * h, a[order], h, label=n,
            color='#6baed6' if n == best_base else None)
ax.set_yticks(np.arange(20)); ax.set_yticklabels([SUPERCLASS_NAMES[i] for i in order])
ax.axvline(chance, color='gray', linestyle=':', label='Chance (1%)')
ax.set_xlabel('Accuracy on that superclass (%)'); ax.set_title('Accuracy by superclass', fontweight='bold')
ax.legend(); ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout(); plt.savefig('cifar100_baseline_superclass.png', dpi=150, bbox_inches='tight'); plt.show()

# ---- plot 5: easiest / hardest classes of the strongest model -----------------------------------
strongest = max(all_preds, key=lambda n: 100 * accuracy_score(y_test, all_preds[n]))
pc = 100 * confusion_matrix(y_test, all_preds[strongest], labels=range(100), normalize='true').diagonal()
o = np.argsort(pc); sel = np.r_[o[:10], o[-10:]]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh([CLASS_NAMES[i] for i in sel], pc[sel], color=['#d62728'] * 10 + ['#2ca02c'] * 10)
ax.set_xlabel('Per-class accuracy (%)'); ax.set_title(f'10 hardest and 10 easiest classes: {strongest}', fontweight='bold')
ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout(); plt.savefig('cifar100_baseline_hardest_easiest.png', dpi=150, bbox_inches='tight'); plt.show()


 65208320/169001437 ━━━━━━━━━━━━━━━━━━━━ 17:28 10us/step

KeyboardInterrupt: 